In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

client = OpenAI()

In [ ]:
SYSTEM_PROMPT = """You are a friendly, practical AI travel advisor.
Help the user plan a personalized, realistic trip through conversation.
Before generating an itinerary, make sure you know all three essentials:
(1) destination, 
(2) trip duration (number of days or dates), and 
(3) at least one travel interest (for example food, museums, hiking, beaches, 
nightlife, or shopping). 
 
Ask one concise, natural follow-up covering only the
missing essentials. Do not ask for optional preferences if the user has not
volunteered them; use sensible assumptions instead.

If the destination, dates, duration, or other details conflict or are materially
ambiguous, ask the user to clarify before planning. Do not silently choose between
contradictory details. Treat an explicit "surprise me" destination as permission
to suggest a destination and proceed, while briefly stating your choice.

Once the essentials are clear and there is no unresolved contradiction, 
provide the complete plan without asking for confirmation. 
Include: 
a memorable trip title; 
a short summary reflecting the user's interests and stated preferences;
a day-by-day itinerary with a sensible pace and geographically coherent activities; 
practical travel tips; 
and an estimated budget with a stated currency,

approximate per-person total, and a clear breakdown 
(such as lodging, food, local transport, and activities). 

State assumptions, especially when dates, departure city,
or budget are unknown. 

Keep estimates approximate, avoid claiming
live availability or current opening hours, and suggest checking details
before booking.

Respect any accessibility, dietary, safety, or budget constraints the user mentions.
Be transparent when a specific recommendation or cost is uncertain.
"""

Token totals are accumulated from each API response's usage data. The cost is an estimate using the Standard `gpt-6-luna` rates listed in the [OpenAI API pricing guide](https://developers.openai.com/api/docs/pricing): $0.10 per 1M input tokens, $0.01 per 1M cached input tokens, and $0.50 per 1M output tokens. Update the rates if you change the model or pricing tier.

In [20]:
MODEL = "gpt-6-luna"

PRICING_USD_PER_MILLION = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50}
}
history = [{"role": "system", "content": SYSTEM_PROMPT}]

conversation_usage = {
    "input_tokens": 0,
    "cached_input_tokens": 0,
    "output_tokens": 0,
    "estimated_cost_usd": 0.0,
}

def send_request(input, **kwargs):
    """Send one Responses API request using the unit's shared model."""
    return client.responses.create(model=MODEL, input=input, **kwargs)

# def get_llm_response(message, system=None):
#     response = client.responses.create(
#         model=MODEL,
#         instructions=system or SYSTEM_PROMPT,
#         input=message,
#     )
#     return response.output_text

# def reset_conversation():
#     history[:] = [{"role": "system", "content": SYSTEM_PROMPT}]
#     conversation_usage.update(
#         input_tokens=0,
#         cached_input_tokens=0,
#         output_tokens=0,
#         estimated_cost_usd=0.0,
#     )

def chat(user_text):
    history.append({"role": "user", "content": user_text})
    response = send_request(history)
    answer = response.output_text
    history.append({"role": "assistant", "content": answer})

    usage = response.usage
    input_tokens = usage.input_tokens
    cached_input_tokens = getattr(
        getattr(usage, "input_tokens_details", None), "cached_tokens", 0
    ) or 0
    output_tokens = usage.output_tokens

    conversation_usage["input_tokens"] += input_tokens
    conversation_usage["cached_input_tokens"] += cached_input_tokens
    conversation_usage["output_tokens"] += output_tokens

    rates = PRICING_USD_PER_MILLION[MODEL]
    uncached_input_tokens = input_tokens - cached_input_tokens
    conversation_usage["estimated_cost_usd"] += (
        uncached_input_tokens * rates["input"]
        + cached_input_tokens * rates["cached_input"]
        + output_tokens * rates["output"]
    ) / 1_000_000

    print(
        f"Conversation totals: {conversation_usage['input_tokens']:,} input tokens, "
        f"{conversation_usage['output_tokens']:,} output tokens; "
        f"estimated cost ${conversation_usage['estimated_cost_usd']:.6f} USD"
    )
    return answer

In [18]:
print(chat("I want to visit Barcelona for 3 days. I love food and architecture."))
print(chat("What should I do on the second day?"))

Conversation totals: 341 input tokens, 1,077 output tokens; estimated cost $0.000573 USD
# Barcelona in Three Days: Modernist Marvels & Catalan Flavors

A relaxed three-day visit pairing Barcelona’s standout architecture with market bites, tapas, and neighborhood restaurants. The plan groups sights by area so you can explore without spending the whole trip in transit.

## Day 1 — Old Barcelona & El Born
- **Morning:** Explore the **Gothic Quarter** on foot: its narrow lanes, Plaça del Rei, and Barcelona Cathedral’s surroundings.
- **Lunch:** Try a neighborhood spot for **pa amb tomàquet** (bread with tomato), seasonal seafood, or a menu del día.
- **Afternoon:** Wander through **El Born**, stopping by the **Basilica of Santa Maria del Mar** and browsing its small shops.
- **Evening:** Have tapas in El Born or nearby Barceloneta. Order a few dishes to share rather than trying to fit in a formal dinner and a tapas crawl.

## Day 2 — Gaudí & the Eixample
- **Morning:** Visit **Sagrada Fam